In [24]:
%load_ext autoreload
%autoreload 2
from girder_client import GirderClient
from dotenv import load_dotenv
import os
import json
import pandas as pd
from augment import get_igsn_xrd_links, extract_alpss_from_portal, download_csv_to_numpy
import io

# --- Load environment variables from .env ---
load_dotenv()

GIRDER_API_URL = os.getenv("GIRDER_API_URL")
API_KEY = os.getenv("HTMDEC_GIRDER_API_KEY")
ALPSS_RESULTS_FOLDER = "692da3649ca0e7d06ecd3d61"
ALPSS_FORM_ID = '6931adb5bb4e23b1e86a1ff1'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
client = GirderClient(apiUrl=GIRDER_API_URL)
client.authenticate(apiKey=API_KEY)
user = client.get("user/me")
print(f"✅ Authenticated as {user['login']}")

✅ Authenticated as arachid1


# XRD

In [28]:
igsn_xrd_links = get_igsn_xrd_links(igsn='JHAMAD00001', client=client)
print(json.dumps(igsn_xrd_links, indent=3))

{
   "scan_point_0_xrd.csv": "https://data.htmdec.org/#item/6931bbe9a27c05f33c328dcd",
   "scan_point_1_xrd.csv": "https://data.htmdec.org/#item/6931bc9dbb4e23b1e86a261d",
   "scan_point_2_xrd.csv": "https://data.htmdec.org/#item/6931bca1bb4e23b1e86a2623",
   "scan_point_3_xrd.csv": "https://data.htmdec.org/#item/6931bca56edf207f78581cfb",
   "scan_point_4_xrd.csv": "https://data.htmdec.org/#item/6931bca86edf207f78581d01"
}


In [27]:
igsn_xrd_csv = download_csv_to_numpy(client, igsn_xrd_links['scan_point_0_xrd.csv'])
print(igsn_xrd_csv.shape)

(1000, 2)


# ALPSS

In [30]:
velocity_time_history, flyer_velocity, spall_strength = extract_alpss_from_portal('C1--20250605--00087', ALPSS_FORM_ID, ALPSS_RESULTS_FOLDER, client)
print("Flyer Velocity:", flyer_velocity)
print("Spall Strength:", spall_strength)

Flyer Velocity: 905.7013045502234
Spall Strength: 19152542751.358006


In [5]:
igsn_velocity_history_csv = download_csv_to_numpy(client, velocity_time_history)
print(igsn_velocity_history_csv.shape)

(19199, 2)


# Curation Example 

In [31]:
path = "test.xlsx"
df = pd.read_excel(path)

In [32]:
df["Sample microstructure/material characterization (EBSD/XRD images)"] = df["Sample_ID"].apply(
    lambda x: get_igsn_xrd_links(x, client)
)

In [33]:
df[[
    "velocity-time history",
    "Flyer velocity",
    "spall strength"
]] = df['PDV_FileName'].apply(
    lambda fn: extract_alpss_from_portal(fn, ALPSS_FORM_ID, ALPSS_RESULTS_FOLDER, client)
).apply(pd.Series)

In [34]:
df.to_excel("augmented_output.xlsx", index=False)